## Carga y limpieza del dataset Online Retail

### Por: Grupo 12 - ITBA
### Fecha: 2026-03-18

### Descripcion:
Carga del dataset crudo, inspeccion de estructura, deteccion de problemas de calidad y generacion de una base limpia lista para analisis.

## Importar librerias

In [1]:
from pathlib import Path

import pandas as pd

## Cargar datos crudos

In [2]:
RAW_PATH = Path("../../data/01_raw/Online Retail.xlsx")
df_raw = pd.read_excel(RAW_PATH)
print(f"Registros: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")
df_raw.head()

Registros: 541,909
Columnas: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Inspeccion general

In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [4]:
df_raw.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [5]:
print("Valores nulos por columna:")
nulls = df_raw.isnull().sum()
nulls_pct = (nulls / len(df_raw) * 100).round(2)
pd.DataFrame({"nulos": nulls, "porcentaje": nulls_pct}).query("nulos > 0")

Valores nulos por columna:


,nulos,porcentaje
Description,1454,0.27
CustomerID,135080,24.93


## Diagnostico de calidad

Problemas detectados:
1. **CustomerID nulo** en ~135k registros (24.9%) - no se puede asignar a un cliente
2. **Description nula** en ~1.5k registros
3. **Cancelaciones**: invoices que empiezan con 'C' tienen Quantity negativa (~9.3k registros)
4. **UnitPrice = 0** en ~2.5k registros (muestras gratuitas, ajustes, etc.)
5. **Tipos mixtos** en StockCode e InvoiceNo (mezcla de strings e integers)

In [6]:
is_cancellation = df_raw["InvoiceNo"].astype(str).str.startswith("C")

print(f"Total registros: {len(df_raw):,}")
print(f"Cancelaciones (InvoiceNo con 'C'): {is_cancellation.sum():,}")
print(f"Quantity negativa: {(df_raw['Quantity'] < 0).sum():,}")
print(f"UnitPrice = 0: {(df_raw['UnitPrice'] == 0).sum():,}")
print(f"UnitPrice negativo: {(df_raw['UnitPrice'] < 0).sum():,}")
print(f"CustomerID nulo: {df_raw['CustomerID'].isnull().sum():,}")

Total registros: 541,909
Cancelaciones (InvoiceNo con 'C'): 9,288
Quantity negativa: 10,624
UnitPrice = 0: 2,515
UnitPrice negativo: 2
CustomerID nulo: 135,080


## Limpieza

Criterios aplicados:
1. Eliminar registros sin CustomerID (no se pueden atribuir a un cliente)
2. Separar cancelaciones en un dataframe aparte para analisis posterior
3. Filtrar registros con Quantity <= 0 y UnitPrice <= 0 del dataframe de ventas
4. Normalizar tipos mixtos en StockCode e InvoiceNo
5. Crear columna Revenue = Quantity * UnitPrice

In [7]:
df = df_raw.copy()

# 1. Eliminar registros sin CustomerID
df = df.dropna(subset=["CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int)
print(f"Despues de eliminar sin CustomerID: {len(df):,}")

# 2. Separar cancelaciones
mask_cancel = df["InvoiceNo"].astype(str).str.startswith("C")
df_cancelaciones = df[mask_cancel].copy()
df = df[~mask_cancel].copy()
print(f"Cancelaciones separadas: {len(df_cancelaciones):,}")
print(f"Ventas validas: {len(df):,}")

# 3. Filtrar Quantity y UnitPrice invalidos
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
print(f"Despues de filtrar qty/price invalidos: {len(df):,}")

# 4. Normalizar tipos mixtos
for col in ["StockCode", "InvoiceNo"]:
    df[col] = df[col].astype(str)
    df_cancelaciones[col] = df_cancelaciones[col].astype(str)

# 5. Crear columna Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
print(f"\nRevenue total: ${df['Revenue'].sum():,.2f}")

Despues de eliminar sin CustomerID: 406,829


Cancelaciones separadas: 8,905
Ventas validas: 397,924


Despues de filtrar qty/price invalidos: 397,884



Revenue total: $8,911,407.90


In [8]:
print("Dataset limpio:")
print(f"  Registros: {len(df):,}")
print(f"  Clientes unicos: {df['CustomerID'].nunique():,}")
print(f"  Invoices unicos: {df['InvoiceNo'].nunique():,}")
print(f"  Productos unicos: {df['StockCode'].nunique():,}")
print(f"  Paises: {df['Country'].nunique()}")
print(f"  Periodo: {df['InvoiceDate'].min().date()} a {df['InvoiceDate'].max().date()}")
df.head()

Dataset limpio:
  Registros: 397,884
  Clientes unicos: 4,338
  Invoices unicos: 18,532
  Productos unicos: 3,665
  Paises: 37
  Periodo: 2010-12-01 a 2011-12-09


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## Guardar datos limpios

In [9]:
PRIMARY_PATH = Path("../../data/03_primary/")
PRIMARY_PATH.mkdir(parents=True, exist_ok=True)

df.to_parquet(PRIMARY_PATH / "ventas_limpias.parquet", index=False)
df_cancelaciones.to_parquet(PRIMARY_PATH / "cancelaciones.parquet", index=False)

print(f"Ventas limpias guardadas: {len(df):,} registros")
print(f"Cancelaciones guardadas: {len(df_cancelaciones):,} registros")

Ventas limpias guardadas: 397,884 registros
Cancelaciones guardadas: 8,905 registros


## Resultados y conclusiones

- El dataset original tiene 541,909 transacciones.
- Se eliminaron ~135k registros sin CustomerID (24.9%) porque no se pueden atribuir a un cliente para segmentacion.
- Se separaron ~8.9k cancelaciones (invoices con prefijo 'C') para analisis aparte.
- Se filtraron registros con Quantity o UnitPrice invalidos.
- La base limpia final tiene ~397k registros de ventas validas con ~4,300 clientes unicos.
- Se creo la columna Revenue para facilitar analisis de facturacion.
- Los datos limpios se guardaron en formato Parquet en `data/03_primary/`.